# minillm 클라우드 학습 (Kaggle / Colab 무료 GPU)

로컬(CPU)에서는 코드 검증만 하고, **실제 학습은 이 노트북에서 무료 GPU로** 돌린다.

**Kaggle 사용법** (권장 — 주 30시간 무료, 세션 12h):
1. kaggle.com → Code → New Notebook
2. 우측 Settings → Accelerator → **GPU T4 x2** (또는 P100)
3. 이 노트북 셀을 위에서부터 실행
4. 세션이 끊기면 마지막 `--resume` 셀만 다시 실행하면 이어서 학습된다.

> `GITHUB_REPO`를 본인 저장소 주소로 바꿔라. 체크포인트는 세션 종료 시
> 사라지므로, 중간중간 `/kaggle/working`의 `checkpoints/`를 **Output으로 저장**하거나
> 마지막 셀로 다운로드해 로컬에 보관한다.

In [ ]:
GITHUB_REPO = 'https://github.com/<YOUR_ID>/minillm.git'  # <-- 본인 저장소로 변경
!git clone $GITHUB_REPO
%cd minillm
!pip install -q regex datasets tqdm
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. 데이터 준비 (최초 1회)
위키 다운로드 → 토크나이저 학습 → 토큰화(.bin). 처음엔 `--max-docs`로 작게 시험하고,
잘 되면 인자를 빼고 전체로 돌린다.

In [ ]:
# 전체를 받으려면 --max-docs 를 지운다 (위키 ko 전체 ≈ 1.4GB, 15~30분)
# vocab 16392 = 기본 특수 토큰 4개 + 마음 유사 기제용 예약 8개 (<|pause|> 등)
!python -m data.download --max-docs 50000
!python -m tokenizer.train_tokenizer --input data/raw/corpus.txt --vocab-size 16392 --sample-mb 200
!python -m data.pack --input data/raw/corpus.txt --tokenizer tokenizer/tokenizer.json

## 2. 사전학습
full 프리셋은 **loop(재귀 깊이)를 켠 채** 학습한다: 중간 4블록을 2회 반복 통과 —
파라미터는 그대로 두고 '생각할 시간'을 준다. 연산이 ~1.5배라 스텝당 시간도 그만큼 는다.
학습 중 반복 횟수를 {1,2}에서 확률적으로 샘플하므로, 완성된 모델은 n_loop=1(빠름)과
n_loop=2(깊음) 어느 쪽으로도 추론할 수 있다.

세션이 끊기면 아래 셀을 `--resume` 붙여 다시 실행.

In [ ]:
!python -m train.pretrain --preset full            # 처음 시작
# !python -m train.pretrain --preset full --resume  # 세션 재개 시 이 줄

### 반복 횟수별 성능 확인
"한 번 더 생각하면 실제로 더 잘 맞히는가?" — 같은 가중치로 n_loop 1/2/3의 val loss를 비교한다.
(3은 학습 범위 밖 외삽 — 보통 나빠지지만 확인해 볼 가치가 있다)

In [ ]:
!python -m tools.eval_loop --ckpt checkpoints/ckpt_best.pt --n-loops 1,2,3

## 3. 대화 파인튜닝 (SFT)
사전학습 best 체크포인트를 이어받아 대화 형식을 가르친다.

마음 유사 기제는 여기서 단계별로 켠다 — 같은 사전학습 체크포인트에서 갈라 학습해
val loss를 비교하는 것이 곧 실험이다:
- `--n-pause 4` : 답변 앞에 <|pause|> 4개 — '생각할 시간'의 가장 싼 형태 (baseline)
- `--mood-dim 64` : 턴 사이에 지속되는 기분 벡터 (FiLM 주입)
- `--latent 2` : Coconut 잠재 사고 — 말하기 전 은닉 상태를 2번 되먹임

In [ ]:
# 기본 + pause 4개 (데이터에 pause를 넣었으면 --n-pause 를 똑같이 준다)
!python -m data.prepare_sft --tokenizer tokenizer/tokenizer.json --n-pause 4
!python -m train.sft --init checkpoints/ckpt_best.pt --data data/bin/sft.npz --out checkpoints/sft_pause4.pt --n-pause 4

# 비교용 baseline (pause 없음)
# !python -m data.prepare_sft --tokenizer tokenizer/tokenizer.json --out data/bin/sft_plain.npz
# !python -m train.sft --init checkpoints/ckpt_best.pt --data data/bin/sft_plain.npz --out checkpoints/sft_plain.pt

# 기분 벡터 (pause 데이터 위에 얹어도 된다)
# !python -m train.sft --init checkpoints/ckpt_best.pt --data data/bin/sft.npz --out checkpoints/sft_mood64.pt --n-pause 4 --mood-dim 64

# Coconut 잠재 사고 (pause 없는 데이터로 — pause와 정면 비교하는 실험)
# !python -m train.sft --init checkpoints/ckpt_best.pt --data data/bin/sft_plain.npz --out checkpoints/sft_latent2.pt --latent 2
# 역피드백 + 확신도 헤드 (메타인지) — mood와 같이 켜도 1-pass를 공유해 비용 동일
# !python -m train.sft --init checkpoints/ckpt_best.pt --data data/bin/sft.npz --out checkpoints/sft_fb_conf.pt --n-pause 4 --feedback --conf
# !python -m tools.eval_conf --ckpt checkpoints/sft_fb_conf.pt --data data/bin/sft.npz  # calibration 확인

## 4. 체크포인트 저장
`checkpoints/sft.pt`와 `tokenizer/tokenizer.json`을 로컬로 내려받아
`python chat.py --ckpt checkpoints/sft.pt`로 대화한다. (Kaggle은 오른쪽 Output 패널에서 다운로드)

In [ ]:
import shutil, os, glob
os.makedirs('/kaggle/working/out', exist_ok=True)
for f in glob.glob('checkpoints/*.pt') + ['tokenizer/tokenizer.json']:
    if os.path.exists(f):
        shutil.copy(f, '/kaggle/working/out/')
print('저장 완료 — Output 패널에서 다운로드하세요')